In [5]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings 
warnings.filterwarnings('ignore')

In [6]:
df=pd.read_csv('UCI_Credit_Card.csv')
df.head(2)

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,1,20000.0,2,2,1,24,2,2,-1,-1,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,2,2,2,26,-1,2,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1


In [7]:
df.drop('ID',axis=1,inplace=True)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 24 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   LIMIT_BAL                   30000 non-null  float64
 1   SEX                         30000 non-null  int64  
 2   EDUCATION                   30000 non-null  int64  
 3   MARRIAGE                    30000 non-null  int64  
 4   AGE                         30000 non-null  int64  
 5   PAY_0                       30000 non-null  int64  
 6   PAY_2                       30000 non-null  int64  
 7   PAY_3                       30000 non-null  int64  
 8   PAY_4                       30000 non-null  int64  
 9   PAY_5                       30000 non-null  int64  
 10  PAY_6                       30000 non-null  int64  
 11  BILL_AMT1                   30000 non-null  float64
 12  BILL_AMT2                   30000 non-null  float64
 13  BILL_AMT3                   300

In [9]:
for col in ['SEX','EDUCATION','MARRIAGE','PAY_0','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6']:
    df[col]=df[col].astype('category')

In [10]:
x=df.drop('default.payment.next.month',axis=1)
y=df['default.payment.next.month']

LOGISTIC REGRESSION WITH SCALED DATA

In [11]:
cat_features = x.select_dtypes(include=['object', 'category']).columns.tolist()
num_features1 = x.select_dtypes(include=['int64', 'float64']).columns.tolist()

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer = StandardScaler()
oh_transformer = OneHotEncoder(drop='first')

preprocessor = ColumnTransformer(
    [
         ("OneHotEncoder", oh_transformer, cat_features),  #Applying one-hot encoding to categorical columns
          ("StandardScaler", numeric_transformer, num_features1) #Applying StandardScaler to numeric columns
    ]
)

In [12]:
x_scaled=preprocessor.fit_transform(x)

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score,accuracy_score,recall_score,f1_score,roc_auc_score

In [14]:
x_train,x_test,y_train,y_test=train_test_split(x_scaled,y,test_size=0.2,random_state=42)
x_train.shape,y_test.shape

((24000, 82), (6000,))

In [15]:
log_reg=LogisticRegression()
log_reg.fit(x_train,y_train)
y_pred=log_reg.predict(x_test)
model_test_accuracy = accuracy_score(y_test, y_pred) # Calculate Accuracy
model_test_f1 = f1_score(y_test, y_pred, average='weighted') # Calculate F1-score
model_test_precision = precision_score(y_test, y_pred) # Calculate Precision
model_test_recall = recall_score(y_test, y_pred) # Calculate Recall
model_test_rocauc_score = roc_auc_score(y_test, y_pred) #Calculate Roc
print(f"Accuracy:{model_test_accuracy:.4f}")
print(f"F1 score:{model_test_f1:.4f}")
print(f"Precision:{model_test_precision:.4f}")
print(f"Recall:{model_test_recall:.4f}")
print(f"ROC:{model_test_rocauc_score:.4f}")

Accuracy:0.8193
F1 score:0.7964
Precision:0.6681
Recall:0.3465
ROC:0.6492


In [19]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold
model = LogisticRegression()

# Define hyperparameter grid
param_grid = {
    'penalty': ['l1', 'l2', 'elasticnet', 'none'],
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['lbfgs', 'saga', 'liblinear'],
    'l1_ratio': [0, 0.5, 1]  # l1_ratio is only used if penalty is 'elasticnet'
}

# Define cross-validation strategy
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Perform grid search with cross-validation
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
grid_search.fit(x, y)

# Best hyperparameters
best_params = grid_search.best_params_
print(f"Best parameters found: {best_params}")



Best parameters found: {'C': 0.01, 'l1_ratio': 0.5, 'penalty': 'l1', 'solver': 'liblinear'}


In [16]:
log_reg=LogisticRegression(C=0.01,l1_ratio=0.5,penalty='l1',solver='liblinear')
log_reg.fit(x_train,y_train)
y_pred=log_reg.predict(x_test)
model_test_accuracy = accuracy_score(y_test, y_pred) # Calculate Accuracy
model_test_f1 = f1_score(y_test, y_pred, average='weighted') # Calculate F1-score
model_test_precision = precision_score(y_test, y_pred) # Calculate Precision
model_test_recall = recall_score(y_test, y_pred) # Calculate Recall
model_test_rocauc_score = roc_auc_score(y_test, y_pred) #Calculate Roc
print(f"Accuracy:{model_test_accuracy:.4f}")
print(f"F1 score:{model_test_f1:.4f}")
print(f"Precision:{model_test_precision:.4f}")
print(f"Recall:{model_test_recall:.4f}")
print(f"ROC:{model_test_rocauc_score:.4f}")

Accuracy:0.8155
F1 score:0.7869
Precision:0.6752
Recall:0.3024
ROC:0.6308


LOGISTIC REGRESSION WITH SCALED DATA AND SMOTE

In [17]:
from imblearn.over_sampling import SMOTE
sm=SMOTE(random_state=42)


In [18]:
x_new,y_new=sm.fit_resample(x_train,y_train)
model1=LogisticRegression()
model1.fit(x_new,y_new)
y_pred_smote=model1.predict(x_test)
model_test_accuracy = accuracy_score(y_test, y_pred_smote) # Calculate Accuracy
model_test_f1 = f1_score(y_test, y_pred_smote, average='weighted') # Calculate F1-score
model_test_precision = precision_score(y_test, y_pred_smote) # Calculate Precision
model_test_recall = recall_score(y_test, y_pred_smote) # Calculate Recall
model_test_rocauc_score = roc_auc_score(y_test, y_pred_smote) #Calculate Roc
print(f"Accuracy:{model_test_accuracy:.4f}")
print(f"F1 score:{model_test_f1:.4f}")
print(f"Precision:{model_test_precision:.4f}")
print(f"Recall:{model_test_recall:.4f}")
print(f"ROC:{model_test_rocauc_score:.4f}")

Accuracy:0.7700
F1 score:0.7781
Precision:0.4793
Recall:0.5903
ROC:0.7053


In [27]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold
model = LogisticRegression()

# Define hyperparameter grid
param_grid = {
    'penalty': ['l1', 'l2', 'elasticnet', 'none'],
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['lbfgs', 'saga', 'liblinear'],
    'l1_ratio': [0, 0.5, 1]  # l1_ratio is only used if penalty is 'elasticnet'
}

# Define cross-validation strategy
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Perform grid search with cross-validation
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
grid_search.fit(x_new, y_new)

# Best hyperparameters
best_params = grid_search.best_params_
print(f"Best parameters found: {best_params}")


Best parameters found: {'C': 1, 'l1_ratio': 0, 'penalty': 'l2', 'solver': 'liblinear'}


In [19]:
log_reg1=LogisticRegression(C=1,l1_ratio=0,penalty='l2',solver='liblinear')
log_reg1.fit(x_new,y_new)
y_pred=log_reg1.predict(x_test)
model_test_accuracy = accuracy_score(y_test, y_pred) # Calculate Accuracy
model_test_f1 = f1_score(y_test, y_pred, average='weighted') # Calculate F1-score
model_test_precision = precision_score(y_test, y_pred) # Calculate Precision
model_test_recall = recall_score(y_test, y_pred) # Calculate Recall
model_test_rocauc_score = roc_auc_score(y_test, y_pred) #Calculate Roc
print(f"Accuracy:{model_test_accuracy:.4f}")
print(f"F1 score:{model_test_f1:.4f}")
print(f"Precision:{model_test_precision:.4f}")
print(f"Recall:{model_test_recall:.4f}")
print(f"ROC:{model_test_rocauc_score:.4f}")

Accuracy:0.7685
F1 score:0.7767
Precision:0.4765
Recall:0.5872
ROC:0.7032


In [20]:
import pickle

In [24]:
with open('logistic.pkl', 'wb') as f:
    pickle.dump(log_reg1, f)

with open('pipeline.pkl', 'wb') as f:
    pickle.dump(preprocessor, f)    